In [4]:
import os
import json
import requests
import time
from dotenv import load_dotenv
from collections import deque

load_dotenv()

steamApiKey = os.getenv("STEAM_API_KEY")
startingSteamId = os.getenv("STEAM_ID")

if not steamApiKey:
    print("API ключ не найден в .env")
    exit()
elif not startingSteamId:
    print("STEAM_ID не найден в .env")
    exit()


def fetchOwnedGamesForUser(steamApiKey: str, userSteamId: str) -> dict:
    url = "http://api.steampowered.com/IPlayerService/GetOwnedGames/v0001/"
    params = {
        "key": steamApiKey,
        "steamid": userSteamId,
        "include_appinfo": True,
        "include_played_free_games": True,
        "format": "json"
    }
    response = requests.get(url, params=params)
    data = response.json()

    responseData = data.get("response", {})
    if "games" not in responseData:
        return {"error": "Нет доступа к играм (профиль приватный или ошибка API)"}
    
    # Оставляем только нужные поля в каждой игре
    slimGames = []
    for game in responseData["games"]:
        slimGames.append({
            "appid": game.get("appid"),
            "name": game.get("name"),
            "playtimeForever": game.get("playtime_forever")
        })
    
    return {
        "games": slimGames,
        "gameCount": responseData.get("game_count", 0)
    }


def fetchPlayerSummaryForUser(steamApiKey: str, userSteamId: str) -> dict:
    url = "http://api.steampowered.com/ISteamUser/GetPlayerSummaries/v0002/"
    params = {
        "key": steamApiKey,
        "steamids": userSteamId,
        "format": "json"
    }
    response = requests.get(url, params=params)
    data = response.json()

    players = data.get("response", {}).get("players", [])
    if not players:
        return {}
    
    player = players[0]
    return {
        "personaname": player.get("personaname"),
        "communityvisibilitystate": player.get("communityvisibilitystate"),
        "timecreated": player.get("timecreated"),
        "loccountrycode": player.get("loccountrycode")
    }


def collectAccountData(
    steamApiKey: str,
    startingUserId: str,
    maxUsers: int = 30,
    maxDepth: int = 2
) -> list:
    dataset = []
    visitedUsers = set()
    queue = deque([(startingUserId, 0)])

    while queue and len(dataset) < maxUsers:
        currentUserId, currentDepth = queue.popleft()
        
        if currentUserId in visitedUsers:
            continue
        visitedUsers.add(currentUserId)

        print(f"Обрабатывается пользователь номер: {len(visitedUsers)} (глубина {currentDepth})...")

        ownedGamesResult = fetchOwnedGamesForUser(steamApiKey, currentUserId)
        playerSummary = fetchPlayerSummaryForUser(steamApiKey, currentUserId)

        userEntry = {
            "steamId": currentUserId,
            "personaname": playerSummary.get("personaname"),
            "communityvisibilitystate": playerSummary.get("communityvisibilitystate"),
            "timecreated": playerSummary.get("timecreated"),
            "loccountrycode": playerSummary.get("loccountrycode"),
            "gameCount": ownedGamesResult.get("gameCount", 0) if "gameCount" in ownedGamesResult else 0,
            "ownedGames": ownedGamesResult.get("games", []) 
                        if "games" in ownedGamesResult 
                        else [],
            "error": ownedGamesResult.get("error") if "error" in ownedGamesResult else None
        }
        dataset.append(userEntry)

        if currentDepth < maxDepth:
            friendsIds = getFriendsListForUser(steamApiKey, currentUserId)
            for friendId in friendsIds[:200]:  # лимит на ветвление
                if friendId not in visitedUsers:
                    queue.append((friendId, currentDepth + 1))
        
        time.sleep(0.0)

    return dataset


def getFriendsListForUser(steamApiKey: str, userSteamId: str) -> list:
    url = "http://api.steampowered.com/ISteamUser/GetFriendList/v0001/"
    params = {
        "key": steamApiKey,
        "steamid": userSteamId,
        "relationship": "friend",
        "format": "json"
    }
    response = requests.get(url, params=params)
    data = response.json()

    friendsList = data.get("friendslist", {}).get("friends", [])
    return [friend["steamid"] for friend in friendsList]


def computeStatistics(dataset: list) -> dict:
    totalUsers = len(dataset)
    usersWithData = [user for user in dataset if not user.get("error")]
    totalGamesOwned = sum(user.get("gameCount", 0) for user in usersWithData)

    gameOwnershipCounter = {}
    for user in usersWithData:
        for game in user.get("ownedGames", []):
            appId = game.get("appid")
            if appId:
                gameOwnershipCounter[appId] = gameOwnershipCounter.get(appId, 0) + 1

    mostPopularGameName = "Нет данных"
    mostPopularCount = 0
    if gameOwnershipCounter:
        mostPopularAppId = max(gameOwnershipCounter, key=gameOwnershipCounter.get)
        mostPopularCount = gameOwnershipCounter[mostPopularAppId]
        for user in usersWithData:
            for game in user.get("ownedGames", []):
                if game.get("appid") == mostPopularAppId:
                    mostPopularGameName = game.get("name", "Unknown")
                    break
            if mostPopularGameName != "Нет данных":
                break

    averageGamesPerUser = round(totalGamesOwned / totalUsers, 2) if totalUsers > 0 else 0

    return {
        "numberOfUsersProcessed": totalUsers,
        "usersWithSuccessfulData": len(usersWithData),
        "totalGamesOwnedAcrossAllUsers": totalGamesOwned,
        "averageGamesOwnedPerUser": averageGamesPerUser,
        "mostPopularGameName": mostPopularGameName,
        "mostPopularGameOwnershipCount": mostPopularCount
    }

In [8]:
print("Запускаем сбор датасета по друзьям...")

collectedDataset = collectAccountData(
    steamApiKey=steamApiKey,
    startingUserId=startingSteamId,
    maxUsers=3,
    maxDepth=2
)

with open("../data/steamUsersDataset.json", "w", encoding="utf-8") as file:
    json.dump(collectedDataset, file, ensure_ascii=False, indent=2)
    print(f"Датасет сохранён в ../data/steamUsersDataset.json ({len(collectedDataset)} пользователей)")

statistics = computeStatistics(collectedDataset)
with open("../data/simpleStatistics.txt", "w", encoding="utf-8") as file:
    file.write(json.dumps(statistics, ensure_ascii=False, indent=2))
    print(f"Статистика сохранена в ../data/simpleStatistics.txt")

Запускаем сбор датасета по друзьям...
Обрабатывается пользователь номер: 1 (глубина 0)...
Обрабатывается пользователь номер: 2 (глубина 1)...
Обрабатывается пользователь номер: 3 (глубина 1)...
Датасет сохранён в ../data/steamUsersDataset.json (3 пользователей)
Статистика сохранена в ../data/simpleStatistics.txt


In [9]:
def fetchGameDetailsFromSteamSpy(steamAppId: int) -> dict:
    url = "https://steamspy.com/api.php"
    params = {
        "request": "appdetails",
        "appid": steamAppId
    }
    try:
        response = requests.get(url, params=params, timeout=15)
        if response.status_code != 200:
            return {"appid": steamAppId, "error": f"HTTP {response.status_code}"}
        
        data = response.json()
        
        return {
            "appid": steamAppId,
            "name": data.get("name"),
            "positive": data.get("positive"),
            "negative": data.get("negative"),
            "owners": data.get("owners"),
            "averageForever": data.get("average_forever"),
            "medianForever": data.get("median_forever"),
            "languages": data.get("languages"),
            "genre": data.get("genre"),
            "tags": data.get("tags", {})
        }
    except Exception as e:
        return {"appid": steamAppId, "error": str(e)}


print("\nНачинаем сбор информации по всем уникальным играм...")

with open("../data/steamUsersDataset.json", "r", encoding="utf-8") as file:
    usersDataset = json.load(file)

uniqueAppIds = set()
for user in usersDataset:
    for game in user.get("ownedGames", []):
        appId = game.get("appid")
        if appId:
            uniqueAppIds.add(appId)

print(f"Найдено {len(uniqueAppIds)} уникальных игр")

gamesDataset = []
for i, appId in enumerate(sorted(uniqueAppIds), 1):
    print(f"[{i}/{len(uniqueAppIds)}] Запрашиваем данные для appid {appId}...")
    gameInfo = fetchGameDetailsFromSteamSpy(appId)
    gamesDataset.append(gameInfo)
    
    time.sleep(0.0)

with open("../data/steamGamesDataset.json", "w", encoding="utf-8") as file:
    json.dump(gamesDataset, file, ensure_ascii=False, indent=2)

print(f"\nСохранено {len(gamesDataset)} игр в ../data/steamGamesDataset.json")


Начинаем сбор информации по всем уникальным играм...
Найдено 501 уникальных игр
[1/501] Запрашиваем данные для appid 10...
[2/501] Запрашиваем данные для appid 20...
[3/501] Запрашиваем данные для appid 50...
[4/501] Запрашиваем данные для appid 70...
[5/501] Запрашиваем данные для appid 80...
[6/501] Запрашиваем данные для appid 100...
[7/501] Запрашиваем данные для appid 130...
[8/501] Запрашиваем данные для appid 220...
[9/501] Запрашиваем данные для appid 240...
[10/501] Запрашиваем данные для appid 280...
[11/501] Запрашиваем данные для appid 320...
[12/501] Запрашиваем данные для appid 360...
[13/501] Запрашиваем данные для appid 400...
[14/501] Запрашиваем данные для appid 440...
[15/501] Запрашиваем данные для appid 500...
[16/501] Запрашиваем данные для appid 550...
[17/501] Запрашиваем данные для appid 620...
[18/501] Запрашиваем данные для appid 730...
[19/501] Запрашиваем данные для appid 1250...
[20/501] Запрашиваем данные для appid 3590...
[21/501] Запрашиваем данные для